# Shrinking the KV Cache: MQA, GQA, and Multi-Head Latent Attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/llm/llm_arch_tricks.ipynb)

Companion notebook to the blog post [Shrinking the KV Cache](https://sesen.ai/blog/recent-llm-architecture-tricks-kv-sharing-mla).

We build one tiny transformer with a **switchable attention block** and train it four times, changing only how key-value heads are shared:

- **MHA** — every query head gets its own KV head
- **GQA** — query heads grouped, each group shares a KV head (Llama-3, Mistral, Gemma)
- **MQA** — all query heads share one KV head (Shazeer 2019)
- **MLA** — cache a single low-rank latent and reconstruct K and V on demand (DeepSeek-V2)

Then we measure the cache size (deterministic) against the validation loss (the part that can bite you).

## Setup

In [ ]:
# Uncomment on Colab / fresh environments
# !pip install torch matplotlib numpy

In [ ]:
import math, json, time, os, urllib.request
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(1337)
device = ('cuda' if torch.cuda.is_available() else
          'mps' if torch.backends.mps.is_available() else 'cpu')
print('Device:', device)

## The data: character-level tiny-shakespeare

A small, honest language-modelling task. The model has to predict the next character, which needs real attention over context.

In [ ]:
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
if not os.path.exists('input.txt'):
    urllib.request.urlretrieve(URL, 'input.txt')
text = open('input.txt').read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size, batch_size = 128, 32

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+1+block_size] for i in ix])
    return x.to(device), y.to(device)

print(f'{len(data):,} chars, vocab {vocab_size}')

## One attention block, four behaviours

The only thing that changes between variants is how K and V are produced and shared. Queries are always per-head.
MHA/GQA/MQA project K and V over `n_kv` heads then `repeat_interleave` them across query heads.
MLA down-projects to a latent `c` (the cached vector) and up-projects K and V from it.

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 65
    block_size: int = 128
    n_layer: int = 4
    n_head: int = 4
    head_dim: int = 32
    attn: str = 'mha'      # mha | gqa | mqa | mla
    n_kv_head: int = 2     # for gqa
    d_latent: int = 32     # for mla

    @property
    def n_embd(self):
        return self.n_head * self.head_dim

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg, self.n_head, self.head_dim = cfg, cfg.n_head, cfg.head_dim
        d = cfg.n_embd
        self.n_kv = {'mha': cfg.n_head, 'mqa': 1,
                     'gqa': cfg.n_kv_head, 'mla': cfg.n_head}[cfg.attn]
        if cfg.attn == 'mla':
            self.W_q  = nn.Linear(d, cfg.n_head * cfg.head_dim, bias=False)
            self.W_dkv = nn.Linear(d, cfg.d_latent, bias=False)        # down-project
            self.W_uk = nn.Linear(cfg.d_latent, cfg.n_head * cfg.head_dim, bias=False)
            self.W_uv = nn.Linear(cfg.d_latent, cfg.n_head * cfg.head_dim, bias=False)
        else:
            self.W_q = nn.Linear(d, cfg.n_head * cfg.head_dim, bias=False)
            self.W_k = nn.Linear(d, self.n_kv * cfg.head_dim, bias=False)
            self.W_v = nn.Linear(d, self.n_kv * cfg.head_dim, bias=False)
        self.proj = nn.Linear(cfg.n_head * cfg.head_dim, d, bias=False)
        self.register_buffer('mask', torch.tril(torch.ones(cfg.block_size, cfg.block_size))
                             .view(1, 1, cfg.block_size, cfg.block_size))

    def forward(self, x):
        B, T, _ = x.shape
        H, hd = self.n_head, self.head_dim
        q = self.W_q(x).view(B, T, H, hd).transpose(1, 2)
        if self.cfg.attn == 'mla':
            c = self.W_dkv(x)                               # (B, T, d_latent) <- cached
            k = self.W_uk(c).view(B, T, H, hd).transpose(1, 2)
            v = self.W_uv(c).view(B, T, H, hd).transpose(1, 2)
        else:
            k = self.W_k(x).view(B, T, self.n_kv, hd).transpose(1, 2)
            v = self.W_v(x).view(B, T, self.n_kv, hd).transpose(1, 2)
            if self.n_kv != H:
                rep = H // self.n_kv
                k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(hd)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        y = (F.softmax(att, dim=-1) @ v).transpose(1, 2).reshape(B, T, H * hd)
        return self.proj(y)

In [ ]:
class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = nn.Sequential(nn.Linear(cfg.n_embd, 4*cfg.n_embd), nn.GELU(),
                                 nn.Linear(4*cfg.n_embd, cfg.n_embd))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
    def forward(self, idx, targets=None):
        T = idx.size(1)
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None]
        for blk in self.blocks:
            x = blk(x)
        logits = self.head(self.ln_f(x))
        loss = None if targets is None else F.cross_entropy(
            logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

## The cache arithmetic

Pure counting. Everything but MLA stores K and V for `n_kv` heads; MLA stores one latent vector.

In [ ]:
def kv_cached_per_token_per_layer(cfg):
    if cfg.attn == 'mha': return 2 * cfg.n_head    * cfg.head_dim
    if cfg.attn == 'gqa': return 2 * cfg.n_kv_head * cfg.head_dim
    if cfg.attn == 'mqa': return 2 * 1             * cfg.head_dim
    if cfg.attn == 'mla': return cfg.d_latent

for a in ['mha', 'gqa', 'mqa', 'mla']:
    cfg = GPTConfig(vocab_size=vocab_size, attn=a)
    e = kv_cached_per_token_per_layer(cfg)
    print(f'{a:4s}: {e:3d} elems/token/layer  ({256//e}x reduction)')

## Train all four variants

Same data, same everything except the attention block. The blog used 3000 steps; we use 1500 here so the notebook finishes quickly. Bump `MAX_STEPS` to 3000 to reproduce the blog numbers exactly.

In [ ]:
MAX_STEPS, EVAL_EVERY, EVAL_ITERS = 1500, 250, 50

@torch.no_grad()
def evaluate(model):
    model.eval()
    losses, correct, total = [], 0, 0
    for _ in range(EVAL_ITERS):
        x, y = get_batch('val')
        logits, loss = model(x, y)
        losses.append(loss.item())
        correct += (logits.argmax(-1) == y).sum().item(); total += y.numel()
    model.train()
    return sum(losses)/len(losses), correct/total

def train(variant):
    cfg = GPTConfig(vocab_size=vocab_size, attn=variant, n_kv_head=2, d_latent=32)
    model = GPT(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.1)
    hist = []
    for step in range(MAX_STEPS + 1):
        if step % EVAL_EVERY == 0:
            vl, va = evaluate(model)
            hist.append((step, vl, va))
        x, y = get_batch('train')
        _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    vl, va = evaluate(model)
    return {'variant': variant, 'loss': vl, 'acc': va,
            'elems': kv_cached_per_token_per_layer(cfg),
            'params': sum(p.numel() for p in model.parameters()), 'hist': hist}

results = {}
for v in ['mha', 'gqa', 'mqa', 'mla']:
    t0 = time.time()
    results[v] = train(v)
    r = results[v]
    print(f"{v:4s} | loss {r['loss']:.4f} | acc {r['acc']:.4f} | "
          f"cache {r['elems']:3d} | {time.time()-t0:.0f}s")

## The trade-off: cache size vs quality

In [ ]:
print(f"{'variant':8s} {'cache':>6s} {'reduction':>10s} {'val_loss':>9s} {'val_acc':>8s}")
for v in ['mha', 'gqa', 'mqa', 'mla']:
    r = results[v]
    print(f"{v:8s} {r['elems']:6d} {256//r['elems']:9d}x {r['loss']:9.4f} {r['acc']:8.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
colors = {'mha':'#1f2937','gqa':'#2563eb','mqa':'#0891b2','mla':'#dc2626'}
for v in ['mha','gqa','mqa','mla']:
    h = results[v]['hist']
    ax1.plot([s for s,_,_ in h], [l for _,l,_ in h], color=colors[v], lw=2, label=v.upper())
ax1.set_xlabel('step'); ax1.set_ylabel('val loss'); ax1.legend(); ax1.grid(alpha=0.3)
ax1.set_title('All four converge together')
for v in ['mha','gqa','mqa','mla']:
    r = results[v]
    ax2.scatter(256//r['elems'], r['loss'], s=160, color=colors[v], zorder=3, edgecolor='white')
    ax2.annotate(v.upper(), (256//r['elems'], r['loss']), xytext=(8,6), textcoords='offset points')
ax2.set_xscale('log', base=2); ax2.set_xlabel('cache reduction (x)'); ax2.set_ylabel('val loss')
ax2.set_title('Smaller cache costs ~1% in loss'); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Exercises

1. **Sweep GQA groups.** Try `n_kv_head` of 1, 2, and 4 and plot loss against cache size. Where is the knee?
2. **Shrink the MLA latent.** Drop `d_latent` to 16 and 8. How small can the latent get before quality breaks? This is the question DeepSeek had to answer at scale.
3. **Scale up.** Push `n_layer`, `n_head`, and `MAX_STEPS` higher. Does the gap between MHA and MQA widen as the model grows? (It should — that is why GQA, not MQA, became the default.)
4. **Add decoupled RoPE.** Our MLA uses learned positional embeddings to keep the latent clean. Real MLA caches a small separate RoPE key. Implement it and confirm the cache size only grows slightly.
5. **Quantise the cache.** Combine an efficient variant with 8-bit quantisation of the cached tensors. Architecture and quantisation reductions multiply.